# CASMI 2026 — RGB → Pilot B (submission experimental)

Este notebook aplica el predictor RGB de espectro→huella y el checkpoint
Pilot B afinado con predicciones **out-of-fold** para generar 25 SMILES por
`molecule_id` del `test.parquet` que Kaggle entregue. Es una ruta experimental:
en 2.000 consultas GEN-VAL, `MRR_novel` fue 0.001504 (IC95% pareado de la
diferencia contra el baseline incluye cero), por debajo del umbral propuesto
de 0.01. No reemplaza el notebook RGB validado.

## Inputs requeridos

1. El input de la competencia, con `test.parquet` (y opcionalmente
   `sample_submission.csv`).
2. El dataset RGB privado, con `W_lineal_rgb20k.npy` y `bins_rgb.npy`.
3. Un input adjunto que contenga el checkpoint **fine-tuned OOF** llamado
   `modelo.pt`. El notebook lo valida por su configuración (`p_rgb_real=0.5`,
   `rgb_top_k=100`) y falla explícitamente si se adjunta el checkpoint base.

Adjunta GPU para acelerar beam search. Internet no es necesario ni se usa;
PyTorch, NumPy, pandas, SciPy y PyArrow deben estar disponibles en el runtime
Kaggle. La salida se escribe en `/kaggle/working/submission.csv`.


In [ ]:
import csv
import hashlib
import os
import re
import resource
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from scipy import sparse

T0 = time.time()
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("working")
WORK.mkdir(parents=True, exist_ok=True)

# Opcionales para probar fuera de Kaggle. En Kaggle se descubren bajo /kaggle/input.
COMPETITION_OVERRIDE = os.environ.get("CASMI_COMPETITION_DIR")
RGB_OVERRIDE = os.environ.get("CASMI_RGB_DIR")
CHECKPOINT_OVERRIDE = os.environ.get("CASMI_B_CHECKPOINT")

def log(msg):
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    print(f"[{time.time() - T0:8.1f}s | RSSmax {rss:7.0f} MB] {msg}", flush=True)

def resolver_directorio(nombre_archivo, override=None):
    if override:
        path = Path(override).expanduser()
        if path.is_file():
            path = path.parent
        if not (path / nombre_archivo).is_file():
            raise FileNotFoundError(f"{path} no contiene {nombre_archivo}")
        return path
    if not INPUT_ROOT.is_dir():
        raise FileNotFoundError(f"no existe {INPUT_ROOT}; adjunta los inputs de Kaggle")
    encontrados = sorted({
        path.parent for path in INPUT_ROOT.rglob(nombre_archivo) if path.is_file()
    })
    if len(encontrados) != 1:
        raise FileNotFoundError(
            f"se esperaba exactamente un {nombre_archivo} bajo {INPUT_ROOT}; "
            f"encontrados: {encontrados}. Define el override correspondiente."
        )
    return encontrados[0]

COMP = resolver_directorio("test.parquet", COMPETITION_OVERRIDE)
RGB = resolver_directorio("W_lineal_rgb20k.npy", RGB_OVERRIDE)
if CHECKPOINT_OVERRIDE:
    CHECKPOINT = Path(CHECKPOINT_OVERRIDE).expanduser()
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(f"no existe checkpoint: {CHECKPOINT}")
else:
    checkpoints = sorted(
        path for path in INPUT_ROOT.rglob("modelo.pt") if path.is_file()
    ) if INPUT_ROOT.is_dir() else []
    if len(checkpoints) != 1:
        raise FileNotFoundError(
            f"se esperaba exactamente un modelo.pt bajo {INPUT_ROOT}; "
            f"encontrados: {checkpoints}. Define CASMI_B_CHECKPOINT."
        )
    CHECKPOINT = checkpoints[0]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("AVISO: no hay GPU; beam search puede tardar mucho.")
else:
    log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"competencia: {COMP} | RGB: {RGB} | checkpoint: {CHECKPOINT}")
log(f"test: {((COMP / 'test.parquet').stat().st_size) / 1e6:.1f} MB")

checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
config = checkpoint["config"]
if float(config.get("p_rgb_real", 0.0)) < 0.5 or config.get("rgb_top_k") != 100:
    raise ValueError(
        "se requiere el checkpoint OOF afinado con p_rgb_real>=0.5 y "
        f"rgb_top_k=100; config recibida: "
        f"p_rgb_real={config.get('p_rgb_real')}, rgb_top_k={config.get('rgb_top_k')}"
    )
log(f"checkpoint OOF validado: n_train={config.get('n_train')}, "
    f"épocas fine-tune={config.get('epocas')}, "
    f"p_rgb_real={config['p_rgb_real']}")


In [ ]:
# Definiciones de tokenizador, modelo y beam search del evaluador B.

NBITS = 2048

SMILES_TOKEN_REGEX = re.compile(
    r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9])"
)

TOKEN_PAD = "<PAD>"

TOKEN_BOS = "<BOS>"

TOKEN_EOS = "<EOS>"

TOKEN_UNK = "<UNK>"

TOKENS_ESPECIALES = [TOKEN_PAD, TOKEN_BOS, TOKEN_EOS, TOKEN_UNK]

def tokenizar_smiles(smiles):
    """SMILES -> lista de tokens. Devuelve (tokens, resto_sin_tokenizar)."""
    tokens = SMILES_TOKEN_REGEX.findall(smiles)
    resto = SMILES_TOKEN_REGEX.sub("", smiles)
    return tokens, resto

class Tokenizador:
    def __init__(self, vocab):
        self.vocab = list(vocab)
        self.tok_a_id = {t: i for i, t in enumerate(self.vocab)}
        self.id_a_tok = {i: t for i, t in enumerate(self.vocab)}
        self.pad = self.tok_a_id[TOKEN_PAD]
        self.bos = self.tok_a_id[TOKEN_BOS]
        self.eos = self.tok_a_id[TOKEN_EOS]
        self.unk = self.tok_a_id[TOKEN_UNK]

    def codificar(self, smiles, max_len):
        toks, _ = tokenizar_smiles(smiles)
        ids = [self.bos] + [self.tok_a_id.get(t, self.unk) for t in toks] + [self.eos]
        return ids[:max_len]

    def decodificar(self, ids):
        toks = []
        for i in ids:
            t = self.id_a_tok.get(int(i), TOKEN_UNK)
            if t in (TOKEN_BOS, TOKEN_PAD):
                continue
            if t == TOKEN_EOS:
                break
            toks.append(t)
        return "".join(toks)

def vector_denso(bits, nbits=NBITS):
    """Bits activos -> vector binario float32 (nbits,)."""
    v = np.zeros(nbits, dtype=np.float32)
    if len(bits):
        v[bits] = 1.0
    return v

def binarizar_predicciones_rgb(predicciones, top_k=100):
    """Convierte scores RGB a huellas binarias con la regla lineal_bin."""
    p = np.asarray(predicciones, dtype=np.float32)
    if p.ndim != 2 or p.shape[1] != NBITS:
        raise ValueError(f"predicciones debe tener forma (n, {NBITS}); recibido {p.shape}")
    if not 1 <= top_k <= NBITS:
        raise ValueError(f"top_k debe estar entre 1 y {NBITS}")
    if not np.isfinite(p).all():
        raise ValueError("predicciones RGB contiene NaN o infinito")
    activas = []
    for fila in p:
        fila = np.clip(fila, 0.0, 1.0)
        kth = top_k if top_k < NBITS else top_k - 1
        idx = np.argpartition(-fila, kth)[:top_k]
        activas.append(np.sort(idx.astype(np.int32)))
    return activas

def _torch():
    try:
        import torch
        import torch.nn as nn
        return torch, nn
    except ImportError as exc:
        raise RuntimeError(
            "torch no está instalado: este script entrena en Colab con GPU. "
            "Instala torch (pip install torch) o corre el notebook de Colab."
        ) from exc

def construir_modelo(vocab_size, d_model=256, nhead=8, num_capas=4,
                     dim_ff=1024, dropout=0.1, k_memoria=8, max_len=160):
    """Transformer encoder-decoder huella->SMILES (torch)."""
    torch, nn = _torch()

    class CodificadorHuella(nn.Module):
        """Huella 2048 bits -> K vectores de memoria (d_model)."""

        def __init__(self):
            super().__init__()
            self.proyeccion = nn.Linear(NBITS, k_memoria * d_model)
            self.pos = nn.Parameter(torch.zeros(1, k_memoria, d_model))
            self.norm = nn.LayerNorm(d_model)

        def forward(self, x):
            b = x.shape[0]
            h = self.proyeccion(x).view(b, k_memoria, d_model)
            return self.norm(h + self.pos)

    class Decodificador(nn.Module):
        def __init__(self):
            super().__init__()
            self.d_model = d_model
            self.k_memoria = k_memoria
            self.max_len = max_len
            self.emb_tokens = nn.Embedding(vocab_size, d_model)
            self.emb_pos = nn.Embedding(max_len, d_model)
            self.cod_huella = CodificadorHuella()
            capa = nn.TransformerDecoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                dropout=dropout, batch_first=True)
            self.decoder = nn.TransformerDecoder(capa, num_layers=num_capas)
            self.salida = nn.Linear(d_model, vocab_size)

        def memoria(self, huella):
            return self.cod_huella(huella)

        def forward(self, huella, tgt_ids, mascara_causal):
            mem = self.memoria(huella)
            pos = torch.arange(tgt_ids.shape[1], device=tgt_ids.device)
            h = self.emb_tokens(tgt_ids) + self.emb_pos(pos)[None, :, :]
            h = self.decoder(h, mem, tgt_mask=mascara_causal)
            return self.salida(h)

    return Decodificador()

def mascara_causal(n, dispositivo):
    torch, nn = _torch()
    return nn.Transformer.generate_square_subsequent_mask(n, device=dispositivo)

def generar_beam(modelo, tok, huella_t, dispositivo, max_len, beam=25):
    """Beam search por lotes (ancho `beam`). Devuelve (listas_smiles,
    listas_scores): por molécula, las `beam` mejores secuencias por
    log-probabilidad acumulada (sin penalización de longitud)."""
    torch, nn = _torch()
    modelo.eval()
    b = huella_t.shape[0]
    # Estado por molécula: lista de (score, [ids]) activos y terminados.
    activos = [[(0.0, [tok.bos])] for _ in range(b)]
    terminados = [[] for _ in range(b)]
    with torch.no_grad():
        for _ in range(max_len - 1):
            # Lote de todas las secuencias activas (misma longitud L).
            planas = []
            dueno = []
            for m in range(b):
                for j, (_, seq) in enumerate(activos[m]):
                    planas.append(seq)
                    dueno.append(m)
            if not planas:
                break
            lote = torch.tensor(planas, dtype=torch.long, device=dispositivo)
            masc = mascara_causal(lote.shape[1], dispositivo)
            # Huellas expandidas al lote (cada secuencia activa lleva la
            # huella de su molécula).
            huella_lote = huella_t[torch.tensor(dueno, device=dispositivo)]
            logits = modelo(huella_lote, lote, masc)
            logp = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
            logp = logp.cpu()
            # Por molécula: top-beam sobre (beam_activo x vocab).
            nuevos_activos = [[] for _ in range(b)]
            for m in range(b):
                idxs = [k for k, d in enumerate(dueno) if d == m]
                if not idxs:
                    continue
                puntajes = torch.tensor(
                    [activos[m][j][0] for j in range(len(activos[m]))])
                cand = puntajes[:, None] + logp[idxs]  # (W, V)
                plano = cand.reshape(-1)
                k = min(beam, plano.numel())
                mejores, pos = torch.topk(plano, k=k)
                for s, p in zip(mejores.tolist(), pos.tolist()):
                    j_beam, tok_id = divmod(p, logp.shape[1])
                    seq = activos[m][j_beam][1] + [tok_id]
                    if tok_id == tok.eos:
                        terminados[m].append((s, seq))
                    else:
                        nuevos_activos[m].append((s, seq))
                # Si todos los beams terminaron, no quedan activos.
                if not nuevos_activos[m] and not terminados[m]:
                    nuevos_activos[m] = activos[m]
            activos = nuevos_activos
            if all(not a for a in activos):
                break
    listas_smiles, listas_scores = [], []
    for m in range(b):
        cands = terminados[m] + activos[m]
        cands.sort(key=lambda x: x[0], reverse=True)
        cands = cands[:beam]
        listas_smiles.append([tok.decodificar(seq) for _, seq in cands])
        listas_scores.append([s for s, _ in cands])
    return listas_smiles, listas_scores

In [ ]:
# Transformación RGB compartida con 21_entrenar_lineal_rgb.py.

def l2_por_fila(X):
    """Igual que en 14: dividir por sqrt(suma de valores de la fila)."""
    norms = np.sqrt(np.asarray(X.sum(axis=1)).ravel().astype(np.float32))
    norms[norms <= 0] = 1.0
    X = sparse.diags((1.0 / norms).astype(np.float32)) @ X
    return X.tocsr().astype(np.float32)

def transformar_rgb(Q, bins_sel):
    """3 canales (sqrt / log1p / binario), cada uno normalizado por fila,
    concatenados [sqrt | log | bin]. Q es la librería bineada cruda."""
    Qb = Q[:, bins_sel].tocsr()
    canales = []
    for modo in ("sqrt", "log", "bin"):
        Xc = Qb.copy().tocsr()
        if modo == "sqrt":
            Xc.data = np.sqrt(Xc.data.astype(np.float32)).astype(np.float32)
        elif modo == "log":
            Xc.data = np.log1p(Xc.data.astype(np.float32)).astype(np.float32)
        else:
            Xc.data = np.ones_like(Xc.data, dtype=np.float32)
        canales.append(l2_por_fila(Xc))
    del Qb
    return sparse.hstack(canales, format="csr").astype(np.float32)

In [ ]:
# Binea cada espectro igual que el runtime RGB, predice scores RGB
# y promedia las salidas por molécula antes de binarizar top-100.
BIN_W = 0.01
NBINS = 200_000
TOPK_PICOS = 100
PMZ_MIN, PMZ_MAX = 100.0, 2000.0

def binear_fila(mzs, intensidades):
    mzs = np.asarray(mzs, dtype=np.float64)
    intensidades = np.asarray(intensidades, dtype=np.float64)
    ok = (
        (intensidades > 0) & np.isfinite(mzs) & np.isfinite(intensidades)
        & (mzs < 2000.0) & (mzs >= 0)
    )
    if not ok.any():
        return None
    mzs, intensidades = mzs[ok], intensidades[ok]
    if len(intensidades) > TOPK_PICOS:
        idx = np.argpartition(-intensidades, TOPK_PICOS)[:TOPK_PICOS]
        mzs, intensidades = mzs[idx], intensidades[idx]
    bins = np.floor(mzs / BIN_W + 1e-9).astype(np.int32)
    bins = np.clip(bins, 0, NBINS - 1)
    ub, inv = np.unique(bins, return_inverse=True)
    vals = np.bincount(inv, weights=intensidades, minlength=len(ub))
    return ub, vals.astype(np.float32)

ids_table = pd.read_parquet(COMP / "test.parquet", columns=["molecule_id"])
if ids_table["molecule_id"].isna().any():
    raise ValueError("test.parquet contiene molecule_id nulos")
test_ids = ids_table["molecule_id"].drop_duplicates().tolist()
del ids_table
parquet = pq.ParquetFile(COMP / "test.parquet")
log(f"test recibido: {parquet.metadata.num_rows:,} espectros, "
    f"{len(test_ids):,} moléculas")

W = np.load(RGB / "W_lineal_rgb20k.npy", mmap_mode="r")
bins_rgb = np.load(RGB / "bins_rgb.npy")
if W.shape != (3 * len(bins_rgb), 2048):
    raise ValueError(f"forma inesperada para W: {W.shape}, bins={len(bins_rgb)}")
id_to_row = {mid: i for i, mid in enumerate(test_ids)}
sums = np.zeros((len(test_ids), 2048), dtype=np.float32)
counts = np.zeros(len(test_ids), dtype=np.int32)
BATCH_ESPECTROS = 8192
n_drop = 0
n_usable = 0
for batch_i, batch in enumerate(parquet.iter_batches(
    batch_size=BATCH_ESPECTROS,
    columns=["molecule_id", "precursor_mz", "ms2_mzs",
             "ms2_normalized_intensities"],
)):
    fields = batch.to_pydict()
    rows, cols, data, batch_ids = [], [], [], []
    for mid, pmz, mzs, intensidades in zip(
        fields["molecule_id"], fields["precursor_mz"], fields["ms2_mzs"],
        fields["ms2_normalized_intensities"],
    ):
        if mid is None or pmz is None or mzs is None or intensidades is None:
            n_drop += 1
            continue
        b = binear_fila(mzs, intensidades)
        if b is None or not PMZ_MIN <= float(pmz) <= PMZ_MAX:
            n_drop += 1
            continue
        bins, vals = b
        row = len(batch_ids)
        rows.extend([row] * len(bins))
        cols.extend(bins.tolist())
        data.extend(vals.tolist())
        batch_ids.append(mid)
    if batch_ids:
        Q = sparse.csr_matrix(
            (np.asarray(data, dtype=np.float32),
             (np.asarray(rows, dtype=np.int32),
              np.asarray(cols, dtype=np.int32))),
            shape=(len(batch_ids), NBINS),
        )
        Q.sum_duplicates()
        Qf = transformar_rgb(Q, bins_rgb)
        del Q
        P_batch = np.asarray(Qf @ W, dtype=np.float32)
        del Qf
        target_rows = np.fromiter(
            (id_to_row[mid] for mid in batch_ids),
            dtype=np.int32, count=len(batch_ids),
        )
        np.add.at(sums, target_rows, P_batch)
        np.add.at(counts, target_rows, 1)
        n_usable += len(batch_ids)
        del P_batch, target_rows
    if (batch_i + 1) % 20 == 0:
        log(f"RGB: lote {batch_i + 1:,}; espectros utilizables "
            f"{n_usable:,}; descartados {n_drop:,}")

bits_by_id = {}
for i in np.flatnonzero(counts):
    scores = (sums[i] / counts[i]).reshape(1, -1)
    bits_by_id[test_ids[i]] = binarizar_predicciones_rgb(scores, top_k=100)[0]
del sums, counts, W
log(f"huellas RGB por molécula: {len(bits_by_id):,}; top-100; "
    f"espectros utilizables {n_usable:,}; descartados {n_drop:,}")

sample_path = COMP / "sample_submission.csv"
if sample_path.is_file():
    output_ids = pd.read_csv(sample_path, usecols=["molecule_id"])[
        "molecule_id"
    ].tolist()
    if set(output_ids) != set(test_ids):
        raise ValueError("IDs de sample_submission.csv no coinciden con test.parquet")
else:
    output_ids = sorted(test_ids, key=str)
if len(set(output_ids)) != len(output_ids):
    raise ValueError("IDs duplicados en la lista de salida")
log(f"filas de salida: {len(output_ids):,}")


In [ ]:
# Genera beam-25 desde la huella RGB top-100. Para GPU de 4 GB se
# conserva lote 1; si hay más memoria puede aumentarse tras una prueba.
BEAM_WIDTH = 25
BATCH_MOLECULAS = 1
if BATCH_MOLECULAS < 1:
    raise ValueError("BATCH_MOLECULAS debe ser >=1")

tokenizer = Tokenizador(checkpoint["vocab"])
model = construir_modelo(
    len(checkpoint["vocab"]),
    d_model=config["d_model"],
    nhead=config["nhead"],
    num_capas=config["num_capas"],
    dim_ff=config["dim_ff"],
    dropout=config["dropout"],
    k_memoria=config["k_memoria"],
    max_len=config["max_len"],
)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)
model.eval()
log(f"modelo Transformer cargado; beam={BEAM_WIDTH}; "
    f"lote={BATCH_MOLECULAS}; max_len={config['max_len']}")

def normalizar_candidatos(cands):
    cands = [s if s else "CCO" for s in cands[:BEAM_WIDTH]]
    if not cands:
        cands = ["CCO"]
    return cands + [cands[0]] * (BEAM_WIDTH - len(cands))

salida = WORK / "submission.csv"
n_sin_espectro = 0
with salida.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["molecule_id", "smiles"])
    for ini in range(0, len(output_ids), BATCH_MOLECULAS):
        ids_lote = output_ids[ini:ini + BATCH_MOLECULAS]
        ids_modelo, huellas = [], []
        for mid in ids_lote:
            bits = bits_by_id.get(mid)
            if bits is None:
                n_sin_espectro += 1
                continue
            ids_modelo.append(mid)
            huellas.append(vector_denso(bits))

        candidatos_por_id = {}
        if huellas:
            huella_t = torch.tensor(np.stack(huellas), dtype=torch.float32,
                                    device=DEVICE)
            listas, _ = generar_beam(
                model, tokenizer, huella_t, DEVICE, config["max_len"],
                beam=BEAM_WIDTH,
            )
            candidatos_por_id = {
                mid: normalizar_candidatos(cands)
                for mid, cands in zip(ids_modelo, listas)
            }

        for mid in ids_lote:
            cands = candidatos_por_id.get(mid, ["CCO"] * BEAM_WIDTH)
            writer.writerow([mid, ";".join(cands)])
        if (ini + len(ids_lote)) % 25 == 0 or ini + len(ids_lote) == len(output_ids):
            log(f"beam search: {min(ini + len(ids_lote), len(output_ids)):,}/"
                f"{len(output_ids):,}")

log(f"submission escrita: {salida}; sin espectro usable: {n_sin_espectro}")


In [ ]:
# Validación estricta del contrato de submission.
submission = pd.read_csv(salida, keep_default_na=False)
n_por_fila = submission["smiles"].map(lambda value: len(value.split(";")))
checks = {
    "columnas": list(submission.columns) == ["molecule_id", "smiles"],
    "filas": len(submission) == len(test_ids),
    "ids_unicos": bool(submission["molecule_id"].is_unique),
    "ids_test": set(map(str, submission["molecule_id"])) == set(map(str, test_ids)),
    "sin_nulos_ni_vacios": bool(
        submission["smiles"].map(lambda value: bool(value.strip())).all()
    ),
    "25_candidatos_por_fila": bool((n_por_fila == BEAM_WIDTH).all()),
}
for nombre, ok in checks.items():
    print(("OK   " if ok else "FAIL ") + nombre)
md5 = hashlib.md5(salida.read_bytes()).hexdigest()
print(f"filas={len(submission):,}; candidatos={int(n_por_fila.sum()):,}; "
      f"md5={md5}; tiempo={time.time() - T0:.1f}s")
assert all(checks.values()), "submission inválida"
print("SUBMISSION VALIDADA")


## Envío

La comprobación final exige una fila por `molecule_id` de test y exactamente
25 candidatos separados por `;`. El archivo queda en
`/kaggle/working/submission.csv`. Adjunta el checkpoint OOF como input
Kaggle antes de ejecutar; el notebook no descarga modelos ni instala paquetes.

El checkpoint usado aquí produjo `MRR_novel` 0.001504 en GEN-VAL, con IC95%
pareado de la diferencia que incluye cero. El submit es exploratorio: no hay
evidencia de que supere la solución RGB actual en el test oculto.
